In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline


In [2]:
text = open('input.txt','r').read()

In [3]:
print(f'length of dataet in characters: {len(text)}')

length of dataet in characters: 1115394


In [4]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [6]:
itos = {i:s for i,s in enumerate(chars)}
stoi = {s:i for i,s in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hello my name is manav"))
print(decode(encode("hello my name is manav")))


[46, 43, 50, 50, 53, 1, 51, 63, 1, 52, 39, 51, 43, 1, 47, 57, 1, 51, 39, 52, 39, 60]
hello my name is manav


In [10]:
data = torch.tensor(encode(text),dtype=torch.long)
print(data.dtype,data.shape)
print(data[:100])

torch.int64 torch.Size([1115394])
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [11]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [14]:
torch.manual_seed(1337)
block_size = 4
batch_size = 8
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data)-block_size,(batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y
xb,yb = get_batch('train')
print('inputs')
print(xb.shape)
print(xb)
print('targets')
print(yb.shape)
print(yb)

print('-----')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b:t+1]
        target = yb[b,t]
        print(f"when input is : {context.tolist()} then output is :{target}")
        
    

inputs
torch.Size([8, 4])
tensor([[ 1, 60, 39, 47],
        [46, 43, 39, 60],
        [ 1, 46, 43, 56],
        [61, 47, 50, 50],
        [43,  1, 39, 52],
        [53, 58, 46,  1],
        [53,  1, 40, 43],
        [ 1, 56, 43, 45]])
targets
torch.Size([8, 4])
tensor([[60, 39, 47, 50],
        [43, 39, 60, 43],
        [46, 43, 56, 43],
        [47, 50, 50,  1],
        [ 1, 39, 52,  1],
        [58, 46,  1, 40],
        [ 1, 40, 43,  1],
        [56, 43, 45, 39]])
-----
when input is : [[1, 60, 39, 47]] then output is :60
when input is : [[1, 60, 39, 47], [46, 43, 39, 60]] then output is :39
when input is : [[1, 60, 39, 47], [46, 43, 39, 60], [1, 46, 43, 56]] then output is :47
when input is : [[1, 60, 39, 47], [46, 43, 39, 60], [1, 46, 43, 56], [61, 47, 50, 50]] then output is :50
when input is : [] then output is :43
when input is : [[46, 43, 39, 60]] then output is :39
when input is : [[46, 43, 39, 60], [1, 46, 43, 56]] then output is :60
when input is : [[46, 43, 39, 60], [1, 46,